# EXP85 Phase 2 Complete Submission

This notebook is an ordered, reproducible front end for the **existing validated Python package**. It imports the implementation from the adjacent `source/`, `scripts/`, and `config/` directories; it does not duplicate or change model logic. Keep this notebook beside those folders. For a submission environment that accepts only one `.ipynb` file, the package must also be bundled or the notebook must be adapted and revalidated there.

**Full training takes several minutes and requires the validated dependencies and CatBoost GPU.** The final execution cell creates a new timestamped run and promotes the submission only after the pipeline's frozen-reference checks pass. The official competition score is not evaluated here.

In [ ]:
from datetime import datetime, timezone
import json
import sys
from pathlib import Path

def find_package_dir():
    expected_name = "EXP85_PHASE2_COMPLETE_SUBMISSION"
    candidates = [Path.cwd(), *(Path.cwd().parents)]
    candidates.extend(
        parent / "champion" / expected_name for parent in [Path.cwd(), *Path.cwd().parents]
    )
    candidates.extend(
        parent / expected_name for parent in [Path.cwd(), *Path.cwd().parents]
    )
    candidates.append(Path(r"D:\\gci\\_gci_push_tmp\\champion\\EXP85_PHASE2_COMPLETE_SUBMISSION"))
    for candidate in candidates:
        if (candidate / "source" / "pipeline.py").is_file() and (candidate / "config" / "model_config.json").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate the Phase 2 package. Open this notebook from its package folder "
        "or clone the repository containing champion/EXP85_PHASE2_COMPLETE_SUBMISSION."
    )

PACKAGE_DIR = find_package_dir()
SOURCE_DIR = PACKAGE_DIR / "source"
SCRIPTS_DIR = PACKAGE_DIR / "scripts"
sys.path[:0] = [str(SOURCE_DIR), str(SCRIPTS_DIR)]

import pandas as pd
from data_validation import load_inputs
from folds import build_folds
from phase1_features import build_decoded_loan_features
from phase2_target_encoding import (
    KEY_SPECS,
    apply_training_encodings,
    key_arrays,
    nested_training_encodings,
)
from prediction_blender import combine_phase2_predictions, mean_fold_predictions
from preprocessing import build_rowwise_features, make_lightgbm_frames
from submission_builder import build_submission, validate_submission
from train_catboost import train_catboost_fold
from train_lightgbm import train_lightgbm_fold
from pipeline import DEFAULT_DATA_DIR, run_pipeline
from verify_pipeline import verify_run

MODEL_CONFIG = json.loads((PACKAGE_DIR / "config" / "model_config.json").read_text(encoding="utf-8"))
FEATURE_CONFIG = json.loads((PACKAGE_DIR / "config" / "feature_config.json").read_text(encoding="utf-8"))
ENSEMBLE_CONFIG = json.loads((PACKAGE_DIR / "config" / "ensemble_config.json").read_text(encoding="utf-8"))
DATA_DIR = DEFAULT_DATA_DIR

print(f"Package: {PACKAGE_DIR}")
print(f"Raw data: {DATA_DIR}")
print("Imported validated package modules; training has not started.")

## 1. Competition data loading and validation

Load the three original CSVs through the package's hash-pinned input loader. It checks the raw input SHA-256 values against the frozen Phase 2 release, validates target/schema/IDs, and confirms the test/sample row order.

In [ ]:
FROZEN_DIR = PACKAGE_DIR.parent / "EXP85_PHASE2_FROZEN_RELEASE"
input_manifest = json.loads(
    (FROZEN_DIR / "HASHES" / "INPUT_HASHES.json").read_text(encoding="utf-8")
)
EXPECTED_INPUT_HASHES = {
    name: record["sha256"]
    for name, record in input_manifest["inputs"].items()
}
train_raw, test_raw, sample_submission, labels, input_checks = load_inputs(
    DATA_DIR, EXPECTED_INPUT_HASHES
)
print(json.dumps({
    "train_rows": input_checks["train_rows"],
    "test_rows": input_checks["test_rows"],
    "sample_rows": input_checks["sample_rows"],
    "class_counts": input_checks["class_counts"],
    "ids_unique": input_checks["ids_unique"],
    "test_ids_equal_sample_order": input_checks["test_ids_equal_sample_order"],
    "input_sha256": input_checks["input_hashes"],
}, indent=2))

## 2. Preprocessing

The production run calls the existing row-wise feature builder and LightGBM categorical-frame converter. They preserve the archived missing-value handling, feature definitions, and train/test column schema. The notebook deliberately does not create a second feature implementation.

In [ ]:
assert callable(build_rowwise_features)
assert callable(make_lightgbm_frames)
print("Preprocessing implementation:", build_rowwise_features.__module__)
print("LightGBM frame implementation:", make_lightgbm_frames.__module__)

## 3. Phase 1 fee/product feature engineering

The pipeline calls the established fee decoder (fee amount 129.5) and product-tier decoder (rate 0.0132). Ratio-frequency counts use pooled train/test covariates without labels, exactly as specified by the frozen recipe.

In [ ]:
assert callable(build_decoded_loan_features)
assert FEATURE_CONFIG["fee_decoder"]["target_used"] is False
assert FEATURE_CONFIG["fee_decoder"]["frequency_counts_use_train_and_test_covariates"] is True
print("Phase 1 implementation:", build_decoded_loan_features.__module__)
print("Fee amount:", FEATURE_CONFIG["fee_decoder"]["fee_amount"])
print("Product rate:", FEATURE_CONFIG["fee_decoder"]["product_rate"])

## 4. Phase 2 nested target encoding

The existing encoder creates inner-fold out-of-fold encodings for outer-training rows. Outer validation and test rows are encoded using only the outer-fit labels. The encoder configuration and formula are read from the package; no alternate encoding logic is introduced here.

In [ ]:
assert callable(nested_training_encodings)
assert callable(apply_training_encodings)
print("Nested target-encoding keys (name, feature, smoothing):")
for key_spec in KEY_SPECS:
    print(" ", key_spec)
print("Inner folds:", MODEL_CONFIG["target_encoding_cv"]["n_splits"])
print("Inner seed:", MODEL_CONFIG["target_encoding_cv"]["seed"])

## 5. Five-fold cross-validation

The production pipeline creates the exact seed-42 stratified folds and checks them against the frozen fold-index evidence before fitting. The next cell only displays the configured split; full fold construction and the exact comparison occur inside the validated runner.

In [ ]:
OUTER_CV = MODEL_CONFIG["outer_cv"]
assert callable(build_folds)
print(json.dumps(OUTER_CV, indent=2))

## 6. LightGBM training

Each outer fold uses the existing CPU LightGBM trainer, archived hyperparameters, fold-specific seed, and early stopping. The full training call below runs all folds once; this section is not a separate fit.

In [ ]:
assert callable(train_lightgbm_fold)
print("LightGBM device:", MODEL_CONFIG["lightgbm"]["device"])
print("Configured rounds:", MODEL_CONFIG["lightgbm"]["num_boost_round"])
print("Early stopping rounds:", MODEL_CONFIG["lightgbm"]["early_stopping_rounds"])

## 7. CatBoost GPU training

Each outer fold uses the existing CatBoost GPU trainer on device 0 with the validated settings and GPU telemetry. The runner performs and records a GPU preflight before model fitting; there is no silent CPU fallback.

In [ ]:
assert callable(train_catboost_fold)
print("CatBoost task type:", MODEL_CONFIG["catboost"]["task_type"])
print("CatBoost device:", MODEL_CONFIG["catboost"]["devices"])
print("Configured iterations:", MODEL_CONFIG["catboost"]["iterations"])

## 8. OOF prediction generation

The runner assigns each validation-fold prediction into its original training-row position, verifies there are no missing/non-finite OOF values, and saves each component's OOF predictions. It separately averages each model's five test-fold predictions.

In [ ]:
assert callable(mean_fold_predictions)
print("OOF and test-fold aggregation are handled by the existing pipeline runner.")

## 9. 50/50 ensemble blending

The frozen ensemble is the arithmetic mean of raw LightGBM and CatBoost class-1 probabilities. No rank transform or post-processing is applied.

In [ ]:
assert callable(combine_phase2_predictions)
assert ENSEMBLE_CONFIG["lightgbm_weight"] == 0.5
assert ENSEMBLE_CONFIG["catboost_weight"] == 0.5
print(ENSEMBLE_CONFIG["formula"])
print("Post-processing:", ENSEMBLE_CONFIG["post_processing"])

## 10. Validation against frozen Phase 2

The full run verifies the frozen release and raw input hashes, requires identical fold assignments, compares OOF AUC and per-fold AUC within documented limits, and checks OOF/test/submission correlations. The final submission is promoted only after every gate passes.

## 11. Final test predictions

For each model, the five fold-specific test probability vectors are averaged; the resulting two vectors are then blended 50/50. The runner saves both component predictions and the final blend.

## 12. Submission CSV generation and verification

Run this cell once to execute the entire validated pipeline. It makes a new timestamped run; existing runs and candidates are never overwritten. The pipeline builds the competition-schema CSV, validates IDs/order and probability bounds, compares against frozen evidence, and promotes only if all checks pass. This is a full training run, not a quick formatting-only operation.

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("notebook_phase2_%Y%m%dT%H%M%S%fZ")
RUN_REPORT = run_pipeline(data_dir=DATA_DIR, mode="full", run_id=RUN_ID)
RUN_DIR = PACKAGE_DIR / "runs" / RUN_ID

assert RUN_REPORT["status"] == "PASS", "Pipeline did not pass its validation gates."
print(json.dumps({
    "run_id": RUN_ID,
    "run_directory": str(RUN_DIR),
    "submission_path": RUN_REPORT["submission"]["promoted_path"],
    "submission_sha256": RUN_REPORT["submission"]["sha256"],
    "oof_auc": RUN_REPORT["ensemble"]["oof_auc"],
    "frozen_oof_auc": RUN_REPORT["ensemble"]["reference_oof_auc"],
    "reproduction_status": RUN_REPORT["reproduction_validation"]["status"],
    "official_competition_score": RUN_REPORT["official_competition_score"],
}, indent=2))

### Optional independent post-run verification

Run after the full pipeline cell. This checks every run-manifest hash, all expected OOF/test prediction artifacts, and the promoted submission against the raw test/sample IDs.

In [ ]:
if "RUN_REPORT" not in globals() or "RUN_DIR" not in globals():
    raise RuntimeError("Run the full pipeline cell before post-run verification.")

RUN_VERIFICATION = verify_run(RUN_DIR, DATA_DIR)
PROMOTED_SUBMISSION = Path(RUN_REPORT["submission"]["promoted_path"])
submission_frame = pd.read_csv(PROMOTED_SUBMISSION)
submission_checks = validate_submission(submission_frame, test_raw, sample_submission)
print(json.dumps({
    "run_verification": RUN_VERIFICATION,
    "submission_verification": submission_checks,
}, indent=2))